# Error Analysis — Multimodal Image Search

Run this after you have a built index (`data/index.faiss`), a manifest, and a labeled query set (`eval/queries.jsonl`). The goal is not just a metric — it's understanding **where retrieval fails and why**, and what you changed in response.

Sections:
1. Load the searcher + queries
2. Overall metrics (Recall@K, mAP)
3. Worst queries (lowest rank of the first relevant hit)
4. Visual inspection of failures
5. Failure taxonomy + conclusions

In [ ]:
import json
import numpy as np
from src.common import load_config
from src.search import Searcher

cfg = load_config('configs/serve.yaml')
searcher = Searcher.from_config(cfg)
queries = [json.loads(l) for l in open('eval/queries.jsonl') if l.strip()]
len(queries)

In [ ]:
# Rank of the first relevant hit per query (None if not retrieved in top-K)
K = 20
rows = []
for q in queries:
    res = searcher.search_text(q['query'], K)
    ranked = [r['id'] for r in res]
    rel = set(q['relevant_ids'])
    first = next((i + 1 for i, rid in enumerate(ranked) if rid in rel), None)
    rows.append({'query': q['query'], 'first_hit_rank': first, 'top': ranked[:5]})

recall_at = lambda k: np.mean([r['first_hit_rank'] is not None and r['first_hit_rank'] <= k for r in rows])
print({f'recall@{k}': round(float(recall_at(k)), 3) for k in (1, 5, 10)})

In [ ]:
# Worst queries: relevant image ranked low or missed entirely
worst = sorted(rows, key=lambda r: (r['first_hit_rank'] is not None, r['first_hit_rank'] or 1e9), reverse=False)
worst = [r for r in rows if r['first_hit_rank'] is None or r['first_hit_rank'] > 5]
for r in worst[:15]:
    print(f"rank={r['first_hit_rank']}  {r['query']}")

In [ ]:
# Visually inspect one failing query's top results vs the ground-truth image
from PIL import Image
import matplotlib.pyplot as plt

q = worst[0]['query'] if worst else queries[0]['query']
res = searcher.search_text(q, 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, r in zip(axes.ravel(), res):
    ax.imshow(Image.open(r['path']).convert('RGB'))
    ax.set_title(f"#{r['id']}  {r['score']:.3f}", fontsize=9)
    ax.axis('off')
fig.suptitle(f'Top results for: {q}')
plt.tight_layout()

## Failure taxonomy + conclusions

Tag each failure and tally the categories — this is the part recruiters care about:

| Category | Example query | Count | Fix tried | Result |
|---|---|---|---|---|
| Domain gap (CLIP unfamiliar) | | | fine-tune / better checkpoint | |
| Ambiguous / underspecified query | | | LLM query rewrite | |
| Fine-grained distinction | | | larger model (ViT-L) | |
| Bad / missing label | | | clean eval set | |

**Conclusion (fill in):** which change moved Recall@K the most, and why.